# Renta Fija: Valoración

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime as dt
import math
from math import log, exp
from typing import Dict
from typing import List, Tuple
import scipy.optimize as optimize

- Hoy podemos invertir en bonos del Tesoro de EE.UU. en distintos vencimientos (3, 5 y 10 años) los cuales ofrecen los siguientes cupones semestrales.
    - 3 años: 3.58 %
    - 5 años: 3.65 %  
    - 10 años: 4.17 %   
- Consideramos que el mercado esta exigiendo una rentabilidad para este tipo de bonos del 3.5%

Determina el valor de los bonos

In [5]:
# Datos de los bonos (par=100)
bonds = {
    "3Y": {"coup": 0.03625, "T": 3},
    "5Y": {"coup": 0.03625, "T": 5},
    "10Y": {"coup":0.04255, "T": 10},
}
ytm=0.035

In [6]:
#Estimamos el valor de los bonos de la reserva federal. Empezamos con el bono a tres años

freq=2 # cupón semestral
par=100 #valor nominal del bono
ytm_f=(1+ytm)**(1/freq)-1 # estimamos la rentabilidad exigida fraccionada k(freq)
periods = bonds["3Y"]["T"]*freq #  número de pagos
coupon = bonds["3Y"]["coup"]*par/freq # valor del cupón
dt = [(i+1) for i in range(int(periods))] # periodos de pago
P3Y = sum([coupon/(1+ytm_f)**(t) for t in dt]) + \
            par/(1+ytm_f)**(periods)
print(f"Precio del bono a 3 años:{P3Y:.2f}")
print(dt)
print(ytm_f)
print(bonds["3Y"]["coup"]/2)

Precio del bono a 3 años:100.44
[1, 2, 3, 4, 5, 6]
0.01734949746879022
0.018125


In [7]:
# Diseñamos una función para no tener que repetirlo
def bond_price(par, T, ytm, coup, freq):
    freq = float(freq)
    ytm_f=(1+ytm)**(1/freq)-1 # estimamos la rentabilidad exigida fraccionada k(freq)
    periods = T*freq #  número de pagos
    coupon = coup*par/freq # valor del cupón
    dt = [(i+1) for i in range(int(periods))] # lista que incluye los exponentes
    price = sum([coupon/(1+ytm_f)**(t) for t in dt]) + \
            par/(1+ytm_f)**(freq*T)
    return price

In [8]:
# Precio de los bonos
for bond, data in bonds.items():
    price = bond_price(par, data["T"], ytm, data["coup"], freq)
    bonds[bond]["price"] = round(price, 2)
bonds

{'3Y': {'coup': 0.03625, 'T': 3, 'price': 100.44},
 '5Y': {'coup': 0.03625, 'T': 5, 'price': 100.71},
 '10Y': {'coup': 0.04255, 'T': 10, 'price': 106.59}}

# Rentabilidad a vencimiento

La rentabilidad a vencimiento (YTM) es la tasa interna de retorno (TIR) que iguala el valor presente de los flujos de efectivo al precio del bono.

In [9]:
P = 950        # precio del bono
VN = 1000      # valor nominal
cupon = 0.06   # 6%
T = 5
freq=2 # semestral

In [10]:
def bond_ytm(price, par, T, coup, freq, guess=0.05): # guess rentabilidad de partida
    freq = float(freq)
    periods = T*freq
    coupon = coup*par/freq
    dt = [(i+1)/freq for i in range(int(periods))]
    ytm_func = lambda y: \
        sum([coupon/(1+y)**(freq*t) for t in dt]) + \
        par/(1+y)**(freq*T) - price

    return (1+optimize.newton(ytm_func, guess))**(freq)-1

In [11]:
y_1=bond_ytm(P,VN,T,cupon,freq)
y_1

np.float64(0.07338662874972712)

Determina la rentabilidad si decido venderlo transcurrido 2 años y los tipos de interés están al 5%.

In [12]:
# Valor del bono transcurrido dos años
ytm_2=0.05
P_2=bond_price(VN, T-2, ytm_2, cupon, freq)
P_2

1029.2500048539573

In [13]:
# Semestres durante dos años
periods=2*freq
dt = [(i+1)/freq for i in range(int(periods))]
print(dt)

[0.5, 1.0, 1.5, 2.0]


In [16]:
coupon = cupon*VN/freq
ytm_func = lambda y: \
        sum([coupon/(1+y)**(freq*t) for t in dt]) + \
        P_2/(1+y)**(freq*2) - P
TIR=(1+optimize.newton(ytm_func, 0.05))**(freq)-1
TIR

np.float64(0.10440663129371575)